# RoBERTa — A Robustly Optimized BERT Pretraining Approach (2019)

_Papers · Transformers & LLMs_

**BERT was undertrained: keep the exact architecture but train it better — more data, longer, bigger batches, fresh random masking every time, and drop the next-sentence task.**

---

This notebook is a practice scaffold for the **RoBERTa — A Robustly Optimized BERT Pretraining Approach (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Cross-entropy at one masked spot

Masked-language-model training predicts the hidden token at each masked position. The loss at one spot is the cross-entropy `-log P(true token)`, where `P` comes from a softmax over the vocabulary. We reproduce the lesson's worked example: logits over a 4-token vocab, true token at index 0, giving a small loss (the model is fairly confident) and the matching perplexity `exp(loss)`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

# worked example: cross-entropy at one masked spot (matches the lesson)
z = torch.tensor([2.0, 1.0, 0.0, 1.0])              # logits over a 4-token vocab
P = z.softmax(0)
print("softmax:", [round(x, 4) for x in P.tolist()]) # ~[0.5345,0.1966,0.0723,0.1966]
ce = -torch.log(P[0])                               # true token = index 0
print("cross-entropy at this spot:", round(ce.item(), 4),
      " perplexity:", round(torch.exp(ce).item(), 4))  # ~0.6264, ~1.871

### Step 2 — Define a tiny BERT-style masked-LM

We build a minimal masked-LM: token + positional embeddings, a small Transformer encoder, and a linear head that predicts over the real vocabulary. Crucially there is **no next-sentence-prediction head** — that is one of RoBERTa's changes. The vocab gets one extra id for the special `[MASK]` token.

In [ ]:
# a tiny BERT-style masked-LM (MLM only — NO NSP head: the RoBERTa choice)
V, D, L, T = 20, 32, 16, 12   # vocab, model dim, seq len, sentences
MASK = V                      # special [MASK] id sits just past the vocab
PAD_IGNORE = -100             # loss ignores these positions

class TinyMLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V + 1, D)           # +1 for [MASK]
        self.pos = nn.Parameter(torch.randn(L, D) * 0.02)
        enc = nn.TransformerEncoderLayer(D, nhead=4, dim_feedforward=64, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
        self.head = nn.Linear(D, V)                 # predict over the V real tokens
    def forward(self, x):
        h = self.emb(x) + self.pos.unsqueeze(0)
        return self.head(self.enc(h))               # (B, L, V) logits — no NSP output

### Step 3 — Implement BERT's 80/10/10 masking

Masking corrupts ~15% of tokens and scores the loss only on those positions. Of the chosen set, 80% become `[MASK]`, 10% become a random token, and the remaining 10% are left unchanged (Section 2.3). The targets tensor uses `-100` everywhere else so cross-entropy ignores unmasked positions. We also fix a held-out masking so both training runs are evaluated on exactly the same task.

In [ ]:
def mask_batch(tokens, rate=0.15):
    """BERT 80/10/10 masking (Section 2.3). Returns corrupted input + targets (-100 = ignore)."""
    x = tokens.clone()
    tgt = torch.full_like(tokens, PAD_IGNORE)
    sel = torch.rand(tokens.shape) < rate           # the masked set M
    tgt[sel] = tokens[sel]                           # score loss only on M
    r = torch.rand(tokens.shape)
    x[sel & (r < 0.8)] = MASK                                       # 80% -> [MASK]
    rand_pos = sel & (r >= 0.8) & (r < 0.9)
    x[rand_pos] = torch.randint(0, V, tokens.shape)[rand_pos]       # 10% -> random
    # remaining 10% (r >= 0.9) left unchanged
    return x, tgt

# fixed toy "corpus" and a held-out masking we score both runs on
corpus = torch.randint(0, V, (T, L))
torch.manual_seed(123)
heldout_x, heldout_t = mask_batch(corpus)   # SAME eval mask for fairness

def held_out_loss(model):
    model.eval()
    with torch.no_grad():
        logits = model(heldout_x)
        return F.cross_entropy(logits.reshape(-1, V), heldout_t.reshape(-1),
                               ignore_index=PAD_IGNORE).item()

### Step 4 — Static vs dynamic masking

This is RoBERTa's headline experiment. **Static** masking freezes one mask pattern and reuses it every epoch (BERT's original approach). **Dynamic** masking re-draws a fresh mask every epoch, so the model practises predicting many different positions and contexts. Holding the model, data, and budget fixed, dynamic masking reaches a slightly lower held-out loss — matching the paper's finding (small-scale, not the paper's exact number).

In [ ]:
def train(dynamic, epochs=60):
    torch.manual_seed(7)                             # same init for both runs
    model = TinyMLM()
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    torch.manual_seed(1)
    static_x, static_t = mask_batch(corpus)          # STATIC: mask once, reuse
    for ep in range(epochs):
        model.train()
        if dynamic:
            x, t = mask_batch(corpus)                # DYNAMIC: fresh mask every epoch
        else:
            x, t = static_x, static_t                # STATIC: frozen pattern
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, V), t.reshape(-1), ignore_index=PAD_IGNORE)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return held_out_loss(model)

static_loss = train(dynamic=False)
dynamic_loss = train(dynamic=True)
print("held-out MLM loss  | static:", round(static_loss, 4),
      " dynamic:", round(dynamic_loss, 4))
print("dynamic better?", dynamic_loss < static_loss,
      "  (our small run, not the paper's number)")

## Visualize the data & results

_Holding the tiny BERT-style masked-LM, the data, and the training budget all fixed, does RoBERTa's dynamic masking (fresh masks every epoch) reach a lower held-out masked-token loss than BERT's static masking (one frozen mask reused every epoch)?_

### Step 1 — Rebuild the tiny masked-LM

The visualization is self-contained, so we restate the model and constants. It is the same MLM-only `TinyMLM` as before — token + positional embeddings, a two-layer Transformer encoder, and a vocab-sized prediction head, with no next-sentence task.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

V, D, L, T = 20, 32, 16, 12
MASK, PAD_IGNORE = V, -100

class TinyMLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V + 1, D)
        self.pos = nn.Parameter(torch.randn(L, D) * 0.02)
        enc = nn.TransformerEncoderLayer(D, nhead=4, dim_feedforward=64, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
        self.head = nn.Linear(D, V)
    def forward(self, x):
        return self.head(self.enc(self.emb(x) + self.pos.unsqueeze(0)))

### Step 2 — Masking, corpus, and held-out evaluation

Same 80/10/10 masking as the reference, plus the fixed toy corpus and a single frozen evaluation mask so both runs are judged on identical held-out positions. The `held_out` helper returns the model's masked-token cross-entropy on that fixed mask.

In [ ]:
def mask_batch(tokens, rate=0.15):
    x = tokens.clone()
    tgt = torch.full_like(tokens, PAD_IGNORE)
    sel = torch.rand(tokens.shape) < rate
    tgt[sel] = tokens[sel]
    r = torch.rand(tokens.shape)
    x[sel & (r < 0.8)] = MASK
    rp = sel & (r >= 0.8) & (r < 0.9)
    x[rp] = torch.randint(0, V, tokens.shape)[rp]
    return x, tgt

corpus = (torch.manual_seed(0), torch.randint(0, V, (T, L)))[1]
torch.manual_seed(123)
hx, ht = mask_batch(corpus)   # fixed eval mask

def held_out(m):
    m.eval()
    with torch.no_grad():
        return F.cross_entropy(m(hx).reshape(-1, V), ht.reshape(-1), ignore_index=PAD_IGNORE).item()

### Step 3 — Train both ways and compare held-out loss

With identical initialization and budget, we train one model with static masking and one with dynamic masking, then print each model's held-out loss. Dynamic comes out slightly lower — the same direction the paper reports, reproduced here at toy scale.

In [ ]:
def run(dynamic, epochs=60):
    torch.manual_seed(7)
    m = TinyMLM()
    opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    torch.manual_seed(1)
    sx, st = mask_batch(corpus)
    for _ in range(epochs):
        m.train()
        x, t = mask_batch(corpus) if dynamic else (sx, st)
        loss = F.cross_entropy(m(x).reshape(-1, V), t.reshape(-1), ignore_index=PAD_IGNORE)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return held_out(m)

print("static :", round(run(False), 3))   # ~2.713
print("dynamic:", round(run(True), 3))    # ~2.546

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** At a masked position the model gives logits $[1.0, 3.0, 0.0]$ over a 3-token vocabulary and the true token is index 1. Compute the cross-entropy at this spot.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Softmax: $e^{1}=2.718$, $e^{3}=20.086$, $e^{0}=1.000$; sum $=23.804$. — _Turns logits into probabilities summing to 1._
- $P(\text{token }1)=20.086/23.804=0.8438$. — _The true token's predicted probability._
- Cross-entropy $=-\log(0.8438)=0.1698$. — _MLM loss at one masked spot is $-\log P(\text{true})$._

**Answer:** About $0.170$. The model is fairly confident in the right token, so the surprise (loss) is small.

</details>

**Problem 2.** ABLATION. You train the tiny masked-LM twice with everything equal except masking: run A masks the same positions every epoch (static), run B re-draws positions every epoch (dynamic). Predict which gives lower held-out masked-token loss, and explain why.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note static practises predicting only the few positions it froze; those other contexts are never trained. — _The masked set $M$ never changes, so coverage of the sentence is narrow._
- Dynamic, over epochs, masks many different positions, so the model practises predicting from many contexts. — _Fresh $M$ each pass = broader, less repetitive training signal._
- Expect dynamic to generalize a touch better on a held-out masking, matching Table 1's 'comparable or slightly better' (Section 4.1). — _Less overfitting to one frozen pattern._

**Answer:** Dynamic should reach a slightly lower held-out loss. The CODE/CODEVIZ run shows exactly this small gap — our small-scale run, not the paper's number. The paper's Table 1 reports the same direction (dynamic SQuAD F1 78.7 vs static 78.3).

</details>

**Problem 3.** Removing NSP: you delete the next-sentence-prediction head and its loss, and instead pack inputs with contiguous full sentences. According to the paper, what happens to downstream performance, and what is the practical benefit?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall Table 2: no-NSP full/doc-sentence formats match or beat segment-pair+NSP (e.g. SQuAD 78.7 → 79.7). — _The paper's controlled comparison (Section 4.2)._
- Conclude the NSP objective was not helping and can be dropped. — _'Removing the NSP loss matches or slightly improves downstream task performance.'_
- Benefit: a simpler objective (MLM only) and inputs that use full context, no wasted capacity on an easy/unhelpful task. — _Less to train, no accuracy lost._

**Answer:** Performance is matched or slightly improved, so NSP is removed. Practically: a simpler, MLM-only recipe with full-sentence inputs — fewer moving parts and at least as good (Section 4.2, Table 2).

</details>